In [1]:
import flwr as fl
from flwr.server.strategy import FedAvg
from flwr.common import NDArrays, Scalar, Metrics
import datasets as hf_ds
from flwr_datasets.partitioner import NaturalIdPartitioner
from flwr_datasets import FederatedDataset



from collections import OrderedDict
from typing import Dict, List, Tuple


import torch
import pandas as pd
import numpy as np
import functools

import sys
sys.path.append('../')


from nwdaf_anomaly_detection.dataloader.dataloader import *
from nwdaf_anomaly_detection.training.training import *
from nwdaf_anomaly_detection.models.rae import *
from nwdaf_anomaly_detection.utils.utils import *
from nwdaf_anomaly_detection.visualizations.visualizations import *
from nwdaf_anomaly_detection.evaluation.evaluation import *

### Load Data and create FederatedDataset

In [2]:
def generate_partitioner(df):
    dataset = hf_ds.Dataset.from_pandas(df)

    partitioner = NaturalIdPartitioner(partition_by = 'imeisv')
    partitioner.dataset = dataset
    
    return partitioner

In [3]:
data_folder = "../Data/Data v5"

In [4]:
data_folder = "../Data/Data v5"
df = pd.read_csv(os.path.join(data_folder, "amari_ue_data_final_v5_smoothed_scaled.csv"))
df = df.sort_values(["imeisv", "_time"], ascending = True)
df['imeisv'] = df['imeisv'].astype(str)
dataset_used = 'smoothed_scaled'

C:\Users\largy\AppData\Local\Temp\ipykernel_26508\2086555185.py:2: DtypeWarning: Columns (16,24,26,27,32,62) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(data_folder, "amari_ue_data_final_v5_smoothed_scaled.csv"))


In [5]:
benign_data_starting_point = "2024-03-20 14:14:50.19"
benign_data_ending_point = "2024-03-23 16:26:19.00"


benign_filter_1 = (df['_time'].between(benign_data_starting_point, benign_data_ending_point))
benign_filter_2 = (~df['imeisv'].isin(['8642840401594200', '8642840401612300','8642840401624200','3557821101183501']))
benign_filter_3 = (df['label'] == 0)
benign_data_filter = (benign_filter_1 & benign_filter_2 & benign_filter_3)

In [6]:
# benign data
benign_data_train = df[benign_data_filter].copy()
benign_data_train = benign_data_train.sort_values(['imeisv','_time'])
print(benign_data_train.shape[0])

260051


In [7]:
benign_data_test_period_start = "2024-03-24 01:20:00.19"
benign_devices_for_testing = ['8609960468879057', '8628490433231157','8677660403123800']

benign_filter_4 = (df['_time'] >= benign_data_test_period_start)
benign_filter_5 = (df['imeisv'].isin(benign_devices_for_testing))
benign_data_filter_test = (benign_filter_3 & benign_filter_4 & benign_filter_5)

benign_data_test = df[benign_data_filter_test].copy()
benign_data_test = benign_data_test.sort_values(['imeisv','_time'])
print(benign_data_test.shape[0])

90102


In [8]:
#malicious data
attck_1_start = "2024-03-23 21:26:00"
attck_1_end = "2024-03-23 22:23:00"
ues_to_exclude_in_1st_attck = [
    '8628490433231157','8609960480666910',
    '3557821101183501'] #'8677660403123800' '8642840401594200'

attck_2_start = "2024-03-23 22:56:00"
attck_2_end = "2024-03-23 23:56:00"
ues_to_exclude_in_2nd_attck = [
    '8609960480666910','8642840401612300'
]

mal_filter_1 = (
    df['_time'].between(attck_1_start, attck_1_end)
    & (~df['imeisv'].isin(ues_to_exclude_in_1st_attck))
)

mal_filter_2 = (
    df['_time'].between(attck_2_start, attck_2_end)
    & (~df['imeisv'].isin(ues_to_exclude_in_2nd_attck))
)

mal_filter_3 = (df['label'] == 1)

malicious_data = df[(mal_filter_1 | mal_filter_2) & mal_filter_3].copy()
malicious_data = malicious_data.sort_values(['imeisv','_time'])
print(malicious_data.shape[0])

10971


In [9]:
benign_train_partitioner = generate_partitioner(benign_data_train)
benign_test_partitioner = generate_partitioner(benign_data_test)
malicious_data_partitioner = generate_partitioner(malicious_data)

In [10]:
benign_train_partitioner.node_id_to_natural_id

{}

In [11]:
dir(benign_train_partitioner)

['__abstractmethods__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_create_int_node_id_to_natural_id',
 '_dataset',
 '_node_id_to_natural_id',
 '_partition_by',
 'dataset',
 'is_dataset_assigned',
 'load_partition',
 'node_id_to_natural_id']

### Define FlowerClient

In [12]:
f = open("../results/experiments_metadata.json")
exp_metadata = json.load(f)
exp_parameters = exp_metadata['5bfa52f8-e8c6-4899-963d-3ebd80be60f9']

In [13]:
class FlowerClient(fl.client.NumPyClient):
    def __init__(
        self,
        exp_parameters, 
        train_data_loader, 
        val_data_loader, 
        mal_data_loader, 
        benign_test_data_loader,
        mal_test_data_loader,
        ) -> None:
        super().__init__()

        self.train_data_loader = train_data_loader, 
        self.val_data_loader = val_data_loader, 
        self.mal_data_loader = mal_data_loader, 
        self.benign_test_data_loader = benign_test_data_loader,
        self.mal_test_data_loader = mal_test_data_loader
        
        self.model =  LSTMAutoencoder(
            input_dim = len(exp_parameters['feature_columns']), 
            hidden_dim1 = exp_parameters['parameters']['hidden_dim1'], 
            hidden_dim2 = exp_parameters['parameters']['hidden_dim2'], 
            output_dim = len(exp_parameters['feature_columns']), 
            dropout = exp_parameters['parameters']['dropout'], 
            layer_norm_flag = exp_parameters['parameters']['layer_norm_flag']
        )
        
        self.criterion = nn.L1Loss() if exp_parameters['loss_function'] == 'L1Loss' else nn.MSELoss()

        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)  # send model to device

    def set_parameters(self, parameters):

        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})

        self.model.load_state_dict(state_dict, strict=True)

    def get_parameters(self, config: Dict[str, Scalar]):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def fit(self, parameters, config):

        self.set_parameters(parameters)

        lr, epochs = config["lr"], config["epochs"]
        
        self.history = self.model.train_model(
            num_epochs = epochs, 
            early_stopping = None, 
            train_data_loader = self.train_data_loader, 
            val_data_loader = self.val_data_loader, 
            mal_data_loader = self.mal_data_loader, 
            device = self.device, 
            criterion = self.criterion,  
            lr = lr
        )

        return self.get_parameters({}), len(self.train_data_loader), {}

    def evaluate(self, parameters: NDArrays, config: Dict[str, Scalar]):

        self.set_parameters(parameters)
        benign_test_losses, mal_test_losses = evaluate_model(
            self.model, 
            self.criterion, 
            self.benign_test_data_loader, 
            self.mal_test_data_loader, 
            self.device
        )
        
        fpr, tpr, thresholds, roc_auc, optimal_threshold = calculate_threshold(benign_test_losses, mal_test_losses)
        accuracy, precision, recall, f1, tp_rate, tn_rate, fp_rate, fn_rate = infer(benign_test_losses, mal_test_losses, optimal_threshold)
        
        return (
            float(np.mean(benign_test_losses)), 
            len(benign_test_losses), 
            {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
            }
        )

### Define Strategy

In [14]:
def fit_config(server_round: int) -> Dict[str, Scalar]:
    config = {
        "epochs": 4, 
        "lr": 0.01,
    }
    return config


def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:

    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]

    return {"accuracy": sum(accuracies) / sum(examples)}

In [15]:
strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0, 
    fraction_evaluate=0.8,
    on_fit_config_fn=fit_config,
    evaluate_metrics_aggregation_fn=weighted_average
)

In [16]:
from torch.utils.data import DataLoader


def get_client_fn():


    def client_fn(cid: str) -> fl.client.Client:
        
        client_benign_train = benign_train_partitioner.load_partition(partition_id=int(cid)).to_pandas()
        client_benign_test = benign_test_partitioner.load_partition(partition_id=int(cid)).to_pandas()
        client_malicious = malicious_data_partitioner.load_partition(partition_id=int(cid)).to_pandas()
        
        train_data_loader, val_data_loader, mal_data_loader = create_ds_loader(
                client_benign_train, 
                client_malicious, 
                exp_parameters['parameters']['window_size'], 
                exp_parameters['parameters']['step_size'], 
                exp_parameters['feature_columns'], 
                exp_parameters['parameters']['batch_size']
            )
        
        test_batch_size = 1
        benign_test_data_loader, mal_test_data_loader = create_test_ds_loaders(
            client_benign_test, 
            client_malicious, 
            exp_parameters['parameters']['window_size'], 
            exp_parameters['parameters']['step_size'], 
            exp_parameters['feature_columns'], 
            batch_size = test_batch_size
        )

        flower_client = FlowerClient(
            exp_parameters, 
            train_data_loader, 
            val_data_loader, 
            mal_data_loader, 
            benign_test_data_loader,
            mal_test_data_loader
        )

        return flower_client.to_client()

    return client_fn


client_fn_callback = get_client_fn()

### Run Simulation

In [17]:
client_resources = {"num_cpus": 1, "num_gpus": 0.1}


history = fl.simulation.start_simulation(
    client_fn=client_fn_callback,
    num_clients=8,
    config=fl.server.ServerConfig(num_rounds=5), 
    strategy=strategy,
    client_resources=client_resources
)

INFO flwr 2024-06-20 11:16:30,604 | app.py:178 | Starting Flower simulation, config: ServerConfig(num_rounds=5, round_timeout=None)
2024-06-20 11:16:36,436	INFO worker.py:1621 -- Started a local Ray instance.
INFO flwr 2024-06-20 11:16:39,566 | app.py:213 | Flower VCE: Ray initialized with resources: {'memory': 904883406.0, 'object_store_memory': 452441702.0, 'node:127.0.0.1': 1.0, 'node:__internal_head__': 1.0, 'CPU': 8.0}
INFO flwr 2024-06-20 11:16:39,567 | app.py:219 | Optimize your simulation with Flower VCE: https://flower.dev/docs/framework/how-to-run-simulations.html
INFO flwr 2024-06-20 11:16:39,567 | app.py:242 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gpus': 0.1}
WARNING flwr 2024-06-20 11:16:39,568 | ray_actor.py:144 | The ActorPool is empty. The system (CPUs=8.0, GPUs=0) does not meet the criteria to host at least one client with resources: {'num_cpus': 1, 'num_gpus': 0.1}. Lowering the `client_resources` could help.


ValueError: ActorPool is empty. Stopping Simulation. Check 'client_resources' passed to `start_simulation`